In [3]:
!pip install z3-solver==4.12.2.0

In [4]:
# ============================================================
# FILTER EXPERIMENT 04 — CLEAN STANDALONE
# A → FilterGate → B pipeline (NONE / HARD / REWRITE)
# MIT-compatible. Zero proprietary references.
# ============================================================

import os
import json
import random
import hashlib
from enum import Enum

import numpy as np
import torch
import torch.nn.functional as F
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# CONFIGURATION
# ============================================================

EXPERIMENT = "Filter Experiment 04"

SEEDS = [11, 22, 33]
EPOCHS = 4
DRAWS_PER_EPOCH = 84

LR = 5e-5
MAX_LENGTH = 64
GRAD_CLIP = 1.0

P_LIE = 0.50
P_UNKNOWN = 0.15

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = "/kaggle/working/filter_exp04"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using device:", DEVICE)


# ============================================================
# EXPECTED FILE HASHES
# ============================================================

EXPECTED_HASHES = {
    "model.safetensors": (
        "c7d00560d8910fbed77ffad4065dee5011c41ba401b1064e749c498ba9e20373"
    ),
    "config.json": (
        "337e7106d8f04da30d6ef1617faa51369cddaa34a855bf76844c9798b46b26e8"
    ),
    "vocab.json": (
        "f6bd25a65e4e63ca31360e9fb11c7e4f9a391a78385d640acd814092dd6eee4f"
    ),
    "merges.txt": (
        "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    ),
}


# ============================================================
# DATA
# ============================================================

GROUND_TRUTH = {
    ("entity_a", "color"): "blue",
    ("entity_b", "color"): "green",
    ("entity_c", "color"): "yellow",
    ("entity_d", "color"): "purple",
    ("entity_e", "shape"): "circle",
    ("entity_f", "shape"): "square",
    ("entity_g", "shape"): "triangle",
    ("entity_h", "shape"): "hexagon",
    ("entity_i", "code"): "17",
    ("entity_j", "code"): "29",
    ("entity_k", "code"): "41",
    ("entity_l", "code"): "53",
}

UNKNOWN_POOL = [
    {
        "subject": "entity_x",
        "predicate": "color",
        "value": "silver",
    },
    {
        "subject": "entity_y",
        "predicate": "shape",
        "value": "star",
    },
]

LIE_BANK = {
    "color": ["red", "orange"],
    "shape": ["triangle", "circle"],
    "code": ["99", "88"],
}

ALLOWED = {
    "color": {
        "blue",
        "green",
        "yellow",
        "purple",
        "red",
        "orange",
        "silver",
    },
    "shape": {
        "circle",
        "square",
        "triangle",
        "hexagon",
        "star",
    },
    "code": {
        "17",
        "29",
        "41",
        "53",
        "99",
        "88",
    },
}

CONTAMINATION_TEST = [
    ("entity_a", "color", "blue", "red"),
    ("entity_b", "color", "green", "orange"),
    ("entity_e", "shape", "circle", "triangle"),
    ("entity_f", "shape", "square", "circle"),
    ("entity_i", "code", "17", "99"),
    ("entity_j", "code", "29", "88"),
]

TRANSFER_TEST = [
    ("entity_z", "shape", "circle", "triangle"),
    ("entity_e", "shape", "circle", "triangle"),
]

UNKNOWN_TEST = [
    ("FACT: entity_x color =", " silver"),
    ("FACT: entity_y shape =", " star"),
]

NEUTRAL_TEXT = "The capital of France is Paris."


# ============================================================
# DATA TYPES
# ============================================================

class Claim(BaseModel):
    subject: str
    predicate: str
    value: str


class Verdict(str, Enum):
    VERIFIED = "VERIFIED"
    CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"
    INVALID = "INVALID"


class Action(str, Enum):
    ADMIT = "ADMIT"
    REJECT = "REJECT"
    REWRITE = "REWRITE"
    HOLD = "HOLD"


# ============================================================
# FILTER GATE
# ============================================================

class FilterGate:
    def __init__(self, ground_truth, policy="hard"):
        if policy not in {"hard", "rewrite", "none"}:
            raise ValueError(
                f"Unknown FilterGate policy: {policy}"
            )

        self.policy = policy

        self.gt = {
            (
                subject.strip().lower(),
                predicate.strip().lower(),
            ): str(value).strip().lower()
            for (subject, predicate), value in ground_truth.items()
        }

    def decide(self, raw):
        try:
            claim = Claim(**raw)
        except Exception:
            return {
                "verdict": Verdict.INVALID.value,
                "action": Action.REJECT.value,
                "payload": None,
            }

        key = (
            claim.subject.strip().lower(),
            claim.predicate.strip().lower(),
        )

        value = claim.value.strip().lower()
        predicate = key[1]

        original = (
            f"FACT: {key[0]} "
            f"{predicate} = {value}"
        )

        # NONE admits every structurally valid generated claim.
        if self.policy == "none":
            return {
                "verdict": self._oracle(key, value),
                "action": Action.ADMIT.value,
                "payload": original,
            }

        # HARD and REWRITE reject values outside the allowed vocabulary.
        if (
            predicate not in ALLOWED
            or value not in ALLOWED[predicate]
        ):
            return {
                "verdict": Verdict.INVALID.value,
                "action": Action.REJECT.value,
                "payload": None,
            }

        # Unknown entities are held rather than trained.
        if key not in self.gt:
            return {
                "verdict": Verdict.UNKNOWN.value,
                "action": Action.HOLD.value,
                "payload": None,
            }

        expected = self.gt[key]

        if value == expected:
            return {
                "verdict": Verdict.VERIFIED.value,
                "action": Action.ADMIT.value,
                "payload": original,
            }

        if self.policy == "rewrite":
            corrected = (
                f"FACT: {key[0]} "
                f"{predicate} = {expected}"
            )

            return {
                "verdict": Verdict.CONTRADICTED.value,
                "action": Action.REWRITE.value,
                "payload": corrected,
            }

        return {
            "verdict": Verdict.CONTRADICTED.value,
            "action": Action.REJECT.value,
            "payload": None,
        }

    def _oracle(self, key, value):
        predicate = key[1]

        if (
            predicate not in ALLOWED
            or value not in ALLOWED[predicate]
        ):
            return Verdict.INVALID.value

        if key not in self.gt:
            return Verdict.UNKNOWN.value

        if value == self.gt[key]:
            return Verdict.VERIFIED.value

        return Verdict.CONTRADICTED.value


# ============================================================
# GENERAL UTILITIES
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sha256_file(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def stable_hash(obj):
    serialized = json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(serialized).hexdigest()


def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# GENERATOR A
# ============================================================

def generator_A(rng):
    # Generate an unknown claim.
    if rng.random() < P_UNKNOWN:
        return dict(
            rng.choice(UNKNOWN_POOL)
        )

    key, truth = rng.choice(
        list(GROUND_TRUTH.items())
    )

    subject, predicate = key

    # Generate a contradicted claim.
    if rng.random() < P_LIE:
        options = [
            value
            for value in LIE_BANK[predicate]
            if value != truth
        ]

        if not options:
            options = list(LIE_BANK[predicate])

        return {
            "subject": subject,
            "predicate": predicate,
            "value": rng.choice(options),
        }

    # Generate a verified claim.
    return {
        "subject": subject,
        "predicate": predicate,
        "value": truth,
    }


def make_A_stream(length, seed):
    rng = random.Random(seed)

    return [
        generator_A(rng)
        for _ in range(length)
    ]


# ============================================================
# EVALUATION FUNCTIONS
# ============================================================

@torch.no_grad()
def continuation_score(model, prefix, continuation):
    model.eval()

    prefix_ids = tokenizer(
        prefix,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(DEVICE)

    full_ids = tokenizer(
        prefix + continuation,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(DEVICE)

    if full_ids.shape[1] < 2:
        raise RuntimeError(
            "The evaluation text contains too few tokens."
        )

    logits = model(
        input_ids=full_ids
    ).logits[:, :-1, :]

    labels = full_ids[:, 1:]

    log_probs = F.log_softmax(
        logits,
        dim=-1,
    )

    selected = torch.gather(
        log_probs,
        dim=-1,
        index=labels.unsqueeze(-1),
    ).squeeze(-1)

    continuation_start = max(
        prefix_ids.shape[1] - 1,
        0,
    )

    continuation_values = selected[
        :,
        continuation_start:,
    ]

    if continuation_values.numel() == 0:
        raise RuntimeError(
            "No continuation tokens were available for scoring."
        )

    return float(
        continuation_values.mean().item()
    )


@torch.no_grad()
def sequence_logprob(model, text):
    model.eval()

    encoded = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(DEVICE)

    input_ids = encoded["input_ids"]

    if input_ids.shape[1] < 2:
        raise RuntimeError(
            "The sequence contains too few tokens."
        )

    logits = model(**encoded).logits

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    log_probs = F.log_softmax(
        shift_logits,
        dim=-1,
    )

    selected = torch.gather(
        log_probs,
        dim=2,
        index=shift_labels.unsqueeze(-1),
    ).squeeze(-1)

    return float(
        selected.mean().item()
    )


def evaluate_pairs(model, label, pairs):
    rows = []

    for subject, predicate, truth, false_value in pairs:
        prefix = (
            f"FACT: {subject} "
            f"{predicate} ="
        )

        truth_score = continuation_score(
            model,
            prefix,
            " " + truth,
        )

        false_score = continuation_score(
            model,
            prefix,
            " " + false_value,
        )

        truth_margin = float(
            truth_score - false_score
        )

        rows.append({
            "branch": label,
            "subject": subject,
            "predicate": predicate,
            "truth": truth,
            "false": false_value,
            "truth_score": float(truth_score),
            "false_score": float(false_score),
            "truth_margin": truth_margin,
            "prefers_truth": bool(truth_margin > 0),
        })

    return rows


def evaluate_unknown(model, label):
    rows = []

    for prefix, value in UNKNOWN_TEST:
        score = continuation_score(
            model,
            prefix,
            value,
        )

        rows.append({
            "branch": label,
            "prefix": prefix,
            "value": value,
            "score": float(score),
        })

    return rows


# ============================================================
# TRAINING LOSS
# ============================================================

def causal_ce(logits, input_ids):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    if shift_labels.numel() == 0:
        raise RuntimeError(
            "The training sequence contains too few tokens."
        )

    return F.cross_entropy(
        shift_logits.view(
            -1,
            shift_logits.size(-1),
        ),
        shift_labels.view(-1),
    )


# ============================================================
# TRAINING BRANCH
# ============================================================

def train_branch(branch, policy, raw_stream, seed):
    set_seed(seed)

    gate = FilterGate(
        GROUND_TRUTH,
        policy,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
    ).to(DEVICE)

    # KV cache is unnecessary during training.
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
    )

    decisions = [
        gate.decide(record)
        for record in raw_stream
    ]

    action_counts = {
        action.value: sum(
            decision["action"] == action.value
            for decision in decisions
        )
        for action in Action
    }

    verdict_counts = {
        verdict.value: sum(
            decision["verdict"] == verdict.value
            for decision in decisions
        )
        for verdict in Verdict
    }

    b_updates = 0
    epoch_size = DRAWS_PER_EPOCH
    epoch_losses = []

    for epoch in range(EPOCHS):
        start = epoch * epoch_size
        end = (epoch + 1) * epoch_size

        raw_chunk = raw_stream[start:end]
        decision_chunk = decisions[start:end]

        chunk = zip(
            raw_chunk,
            decision_chunk,
        )

        current_epoch_losses = []
        current_epoch_updates = 0

        model.train()

        for raw_record, decision in chunk:
            payload = decision["payload"]

            if payload is None:
                continue

            encoded = tokenizer(
                payload,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(**encoded).logits

            loss = causal_ce(
                logits,
                encoded["input_ids"],
            )

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"{branch}: non-finite loss "
                    f"at epoch {epoch + 1}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            loss_value = float(
                loss.detach().cpu().item()
            )

            current_epoch_losses.append(
                loss_value
            )

            b_updates += 1
            current_epoch_updates += 1

        if current_epoch_losses:
            mean_epoch_loss = float(
                np.mean(current_epoch_losses)
            )
        else:
            mean_epoch_loss = None

        epoch_losses.append({
            "epoch": int(epoch + 1),
            "updates": int(current_epoch_updates),
            "mean_loss": mean_epoch_loss,
        })

        if mean_epoch_loss is None:
            loss_text = "N/A"
        else:
            loss_text = f"{mean_epoch_loss:.6f}"

        print(
            f"Seed {seed} | "
            f"{branch} | "
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Updates {current_epoch_updates} | "
            f"Loss {loss_text}"
        )

    contamination_results = evaluate_pairs(
        model,
        branch,
        CONTAMINATION_TEST,
    )

    transfer_results = evaluate_pairs(
        model,
        branch,
        TRANSFER_TEST,
    )

    unknown_results = evaluate_unknown(
        model,
        branch,
    )

    neutral_logprob = sequence_logprob(
        model,
        NEUTRAL_TEXT,
    )

    results = {
        "b_updates": int(b_updates),
        "action_counts": action_counts,
        "verdict_counts": verdict_counts,
        "epoch_losses": epoch_losses,
        "contamination": contamination_results,
        "transfer": transfer_results,
        "unknown": unknown_results,
        "neutral_logprob": float(neutral_logprob),
    }

    # The optimizer also keeps references to model parameters.
    del optimizer
    del model

    clear_memory()

    return results


# ============================================================
# LOCATE MODEL AND TOKENIZER
# ============================================================

model_candidates = []
tokenizer_candidates = []

for root, _, files in os.walk("/kaggle/input"):
    file_set = set(files)

    if {
        "config.json",
        "model.safetensors",
    }.issubset(file_set):
        model_candidates.append(root)

    if {
        "vocab.json",
        "merges.txt",
    }.issubset(file_set):
        tokenizer_candidates.append(root)

if not model_candidates:
    raise FileNotFoundError(
        "No directory containing config.json and "
        "model.safetensors was found under /kaggle/input."
    )

if not tokenizer_candidates:
    raise FileNotFoundError(
        "No directory containing vocab.json and "
        "merges.txt was found under /kaggle/input."
    )

MODEL_PATH = model_candidates[0]
TOKENIZER_PATH = tokenizer_candidates[0]

print("Model path:", MODEL_PATH)
print("Tokenizer path:", TOKENIZER_PATH)


# ============================================================
# VERIFY HASHES
# ============================================================

for filename, expected_hash in EXPECTED_HASHES.items():
    if filename in {
        "model.safetensors",
        "config.json",
    }:
        file_path = os.path.join(
            MODEL_PATH,
            filename,
        )
    else:
        file_path = os.path.join(
            TOKENIZER_PATH,
            filename,
        )

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"Required file was not found: {file_path}"
        )

    actual_hash = sha256_file(file_path)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Hash mismatch: {filename}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )

    print(f"Hash verified: {filename}")


# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# BASE MODEL EVALUATION
# ============================================================

print("\nEvaluating base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
).to(DEVICE)

base_contamination = evaluate_pairs(
    base_model,
    "BASE",
    CONTAMINATION_TEST,
)

base_transfer = evaluate_pairs(
    base_model,
    "BASE",
    TRANSFER_TEST,
)

base_unknown = evaluate_unknown(
    base_model,
    "BASE",
)

base_neutral = sequence_logprob(
    base_model,
    NEUTRAL_TEXT,
)

base_results = {
    "contamination": base_contamination,
    "transfer": base_transfer,
    "unknown": base_unknown,
    "neutral_logprob": float(base_neutral),
    "mean_margin": float(
        np.mean([
            row["truth_margin"]
            for row in base_contamination
        ])
    ),
}

del base_model
clear_memory()

print(
    "Base mean contamination margin:",
    base_results["mean_margin"],
)


# ============================================================
# RUN EXPERIMENT
# ============================================================

BRANCHES = {
    "NONE": "none",
    "HARD": "hard",
    "REWRITE": "rewrite",
}

all_runs = []

for seed in SEEDS:
    print("\n" + "=" * 65)
    print(f"STARTING SEED: {seed}")
    print("=" * 65)

    raw_stream = make_A_stream(
        EPOCHS * DRAWS_PER_EPOCH,
        seed,
    )

    branch_runs = {}

    # All branches receive the same raw stream for this seed.
    for branch_name, policy in BRANCHES.items():
        print(
            f"\nTraining branch {branch_name} "
            f"with policy {policy}"
        )

        branch_runs[branch_name] = train_branch(
            branch_name,
            policy,
            raw_stream,
            seed,
        )

    summary = {}

    for branch_name in BRANCHES:
        contamination = branch_runs[
            branch_name
        ]["contamination"]

        transfer = branch_runs[
            branch_name
        ]["transfer"]

        entity_z_margins = [
            row["truth_margin"]
            for row in transfer
            if row["subject"] == "entity_z"
        ]

        if not entity_z_margins:
            raise RuntimeError(
                f"No entity_z transfer result for {branch_name}."
            )

        summary[branch_name] = {
            "mean_margin": float(
                np.mean([
                    row["truth_margin"]
                    for row in contamination
                ])
            ),
            "mean_entity_z_margin": float(
                np.mean(entity_z_margins)
            ),
            "b_updates": int(
                branch_runs[
                    branch_name
                ]["b_updates"]
            ),
            "neutral_logprob": float(
                branch_runs[
                    branch_name
                ]["neutral_logprob"]
            ),
        }

    print(
        f"\nSeed {seed} summary:"
    )

    print(
        json.dumps(
            summary,
            indent=2,
            ensure_ascii=False,
        )
    )

    all_runs.append({
        "seed": int(seed),
        "raw_stream_hash": stable_hash(
            raw_stream
        ),
        "summary": summary,
        "branches": branch_runs,
    })


# ============================================================
# AGGREGATE RESULTS
# ============================================================

aggregate = {
    "runs": int(len(SEEDS)),
    "seeds": [
        int(seed)
        for seed in SEEDS
    ],
}

for branch_name in BRANCHES:
    margin_values = [
        run["summary"][branch_name]["mean_margin"]
        for run in all_runs
    ]

    entity_z_values = [
        run["summary"][branch_name]["mean_entity_z_margin"]
        for run in all_runs
    ]

    update_values = [
        run["summary"][branch_name]["b_updates"]
        for run in all_runs
    ]

    neutral_values = [
        run["summary"][branch_name]["neutral_logprob"]
        for run in all_runs
    ]

    aggregate[branch_name] = {
        "mean_margin": float(
            np.mean(margin_values)
        ),
        "std_margin": float(
            np.std(
                margin_values,
                ddof=1,
            )
        ),
        "mean_entity_z_margin": float(
            np.mean(entity_z_values)
        ),
        "mean_b_updates": float(
            np.mean(update_values)
        ),
        "mean_neutral_logprob": float(
            np.mean(neutral_values)
        ),
    }


# ============================================================
# PAIRED BRANCH DIFFERENCES
# ============================================================

hard_vs_none = np.array([
    run["summary"]["HARD"]["mean_margin"]
    for run in all_runs
]) - np.array([
    run["summary"]["NONE"]["mean_margin"]
    for run in all_runs
])

rewrite_vs_none = np.array([
    run["summary"]["REWRITE"]["mean_margin"]
    for run in all_runs
]) - np.array([
    run["summary"]["NONE"]["mean_margin"]
    for run in all_runs
])

rewrite_vs_hard = np.array([
    run["summary"]["REWRITE"]["mean_margin"]
    for run in all_runs
]) - np.array([
    run["summary"]["HARD"]["mean_margin"]
    for run in all_runs
])

aggregate["HARD_vs_NONE"] = {
    "mean": float(
        np.mean(hard_vs_none)
    ),
    "std": float(
        np.std(
            hard_vs_none,
            ddof=1,
        )
    ),
}

aggregate["REWRITE_vs_NONE"] = {
    "mean": float(
        np.mean(rewrite_vs_none)
    ),
    "std": float(
        np.std(
            rewrite_vs_none,
            ddof=1,
        )
    ),
}

aggregate["REWRITE_vs_HARD"] = {
    "mean": float(
        np.mean(rewrite_vs_hard)
    ),
    "std": float(
        np.std(
            rewrite_vs_hard,
            ddof=1,
        )
    ),
}


# ============================================================
# SAVE RESULTS
# ============================================================

result = {
    "experiment": EXPERIMENT,
    "status": "completed",
    "device": str(DEVICE),
    "seeds": [
        int(seed)
        for seed in SEEDS
    ],
    "configuration": {
        "epochs": int(EPOCHS),
        "draws_per_epoch": int(DRAWS_PER_EPOCH),
        "learning_rate": float(LR),
        "max_length": int(MAX_LENGTH),
        "gradient_clip": float(GRAD_CLIP),
        "p_lie": float(P_LIE),
        "p_unknown": float(P_UNKNOWN),
    },
    "base": base_results,
    "aggregate": aggregate,
    "runs": all_runs,
    "independent": True,
    "license_suggested": "MIT",
}

output_file = os.path.join(
    OUTPUT_DIR,
    "filter_exp04_results.json",
)

with open(
    output_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("\n" + "=" * 65)
print("EXP04 COMPLETE")
print("Output file:", output_file)
print("Aggregate results:")
print(
    json.dumps(
        aggregate,
        indent=2,
        ensure_ascii=False,
    )
)
print("SHA256:", sha256_file(output_file))
print("=" * 65)


Using device: cpu
Model path: /kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_model/gpt2_model
Tokenizer path: /kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_tokenizer/gpt2_tokenizer
Hash verified: model.safetensors
Hash verified: config.json
Hash verified: vocab.json
Hash verified: merges.txt

Evaluating base model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Base mean contamination margin: -0.1503168741861979

STARTING SEED: 11

Training branch NONE with policy none


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 11 | NONE | Epoch 1/4 | Updates 84 | Loss 1.616456
Seed 11 | NONE | Epoch 2/4 | Updates 84 | Loss 0.818424
Seed 11 | NONE | Epoch 3/4 | Updates 84 | Loss 0.753783
Seed 11 | NONE | Epoch 4/4 | Updates 84 | Loss 0.631332

Training branch HARD with policy hard


...

Seed 33 | NONE | Epoch 1/4 | Updates 84 | Loss 1.604155
Seed 33 | NONE | Epoch 2/4 | Updates 84 | Loss 0.817533
Seed 33 | NONE | Epoch 3/4 | Updates 84 | Loss 0.769853
Seed 33 | NONE | Epoch 4/4 | Updates 84 | Loss 0.631085

Training branch HARD with policy hard


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 33 | HARD | Epoch 1/4 | Updates 45 | Loss 1.895438
Seed 33 | HARD | Epoch 2/4 | Updates 44 | Loss 0.816791
Seed 33 | HARD | Epoch 3/4 | Updates 30 | Loss 0.751086
Seed 33 | HARD | Epoch 4/4 | Updates 37 | Loss 0.735834

Training branch REWRITE with policy rewrite


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 33 | REWRITE | Epoch 1/4 | Updates 69 | Loss 1.742328
Seed 33 | REWRITE | Epoch 2/4 | Updates 74 | Loss 0.768771
Seed 33 | REWRITE | Epoch 3/4 | Updates 70 | Loss 0.592866
Seed 33 | REWRITE | Epoch 4/4 | Updates 71 | Loss 0.491257

Seed 33 summary:
{
  "NONE": {
    "mean_margin": 0.7184702654679617,
    "mean_entity_z_margin": -0.08309173583984375,
    "b_updates": 336,
    "neutral_logprob": -10.399781227111816
  },
  "HARD": {
    "mean_margin": 4.987073217829068,
    "mean_entity_z_margin": -3.289482355117798,
    "b_updates": 156,
    "neutral_logprob": -9.063812255859375
  },
  "REWRITE": {
    "mean_margin": 12.728368207402431,
    "mean_entity_z_margin": -0.9993438720703125,
    "b_updates": 284,
    "neutral_logprob": -10.142566680908203
  }
}

EXP04 COMPLETE
Output file: /kaggle/working/filter_exp04/filter_exp04_results.json
Aggregate results:
{
  "runs": 3,
  "seeds": [
    11,
    22,
    33
  ],
  "NONE": {
    "mean_margin": -0.014722598095734877,
    "std_margin": 0